# Module 18 lab · Pretraining at scale

The code from every lesson of [this module](https://aicourses.dev/research/pretraining-at-scale), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**This lab needs a GPU.** Colab opens it on one; if not, choose Runtime → Change runtime type → T4 GPU.

**First time?** Read the [one-time setup](https://aicourses.dev/research/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 18.2 Inside a GPU: memory hierarchy, tensor cores and the roofline

A GPU has a little very fast memory and a lot of slower memory, and special units for matrix maths. The roofline is a chart that says whether a step is slowed by maths or by moving data, the question behind every speed-up.

[Read the lesson](https://aicourses.dev/research/pretraining-at-scale/inside-a-gpu)

### The roofline in code

The roofline is plain arithmetic, so you can compute it anywhere:

In [ ]:
def intensity(flops, bytes_moved):
    return flops / bytes_moved

n = 4096
# vector add in BF16: read 2 numbers, write 1, each 2 bytes
print(intensity(n, 3 * 2 * n))            # 0.16666666666666666
# n-by-n matmul in BF16: 2n^3 FLOPs, three matrices moved
print(intensity(2 * n**3, 3 * 2 * n * n))  # 1365.3333333333333

peak = 989e12        # H100 SXM, dense BF16, FLOP/s
bandwidth = 3.35e12  # H100 SXM memory, bytes/s
print(round(peak / bandwidth))   # 295, the ridge point

def attainable(i):
    return min(peak, bandwidth * i)

for name, i in [("add", 1 / 6), ("matmul", n / 3)]:
    print(name, f"{attainable(i) / 1e12:.2f} TFLOP/s")
# add 0.56 TFLOP/s
# matmul 989.00 TFLOP/s

The vector add can reach 0.56 trillion FLOPs a second, about 0.06% of the peak, the puzzle from the start. A CPU follows the same rule. This times an add and a matmul on the same two 4,096 × 4,096 matrices, on Colab’s CPU runtime:

In [ ]:
import time
import torch

torch.manual_seed(0)
n = 4096
a, b = torch.randn(n, n), torch.randn(n, n)

def speed(fn, flops, repeats=10):
    fn()                                  # warm-up
    start = time.perf_counter()
    for _ in range(repeats):
        fn()
    seconds = (time.perf_counter() - start) / repeats
    return flops / seconds / 1e9          # GFLOP/s

print(f"add:    {speed(lambda: a + b, n * n):.1f} GFLOP/s")
print(f"matmul: {speed(lambda: a @ b, 2 * n**3):.1f} GFLOP/s")

## 18.3 Profiling and fixing a slow training step

The GPU runs small programs called kernels while Python moves on without waiting, so timing with a clock is wrong. A profiler records what really ran and finds the slow part. It also gives your MFU: the share of the GPU’s top speed your run actually uses.

[Read the lesson](https://aicourses.dev/research/pretraining-at-scale/profiling)

### Kernels, and why Python does not wait

To time GPU work, wait for it to finish. `torch.cuda.synchronize()` makes Python wait until the queue is empty, and **CUDA events** are timestamps the GPU itself records in its queue (CUDA is NVIDIA’s software for programming its GPUs):

In [ ]:
import torch

x = torch.randn(8192, 8192, device="cuda")
torch.cuda.synchronize()            # finish earlier work first
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)
start.record()                      # timestamp in the GPU's queue
y = x @ x
end.record()
torch.cuda.synchronize()            # wait until the matmul is done
print(start.elapsed_time(end), "ms")

### Three kinds of slow

You can see overhead without a GPU. Here the same matmul work runs two ways on Colab’s CPU runtime: one row at a time in a Python loop, and as one batched call.

In [ ]:
import time
import torch

torch.manual_seed(0)
layer = torch.nn.Linear(512, 512)
x = torch.randn(256, 512)

def slow_step():            # one row at a time
    return torch.stack([layer(row) for row in x])

def fast_step():            # all rows in one call
    return layer(x)

def ms(fn, repeats=100):
    fn()                    # warm-up
    start = time.perf_counter()
    for _ in range(repeats):
        fn()
    return (time.perf_counter() - start) / repeats * 1000

print(torch.allclose(slow_step(), fast_step(), atol=1e-6))  # True
print(f"slow: {ms(slow_step):.2f} ms, fast: {ms(fast_step):.2f} ms")

### Reading a profile

`torch.profiler` records every operation that ran. On the CPU you can use it straight away:

In [ ]:
from torch.profiler import profile, ProfilerActivity

for step in (slow_step, fast_step):
    with profile(activities=[ProfilerActivity.CPU]) as prof:
        step()
    ops = prof.key_averages()
    total = sum(e.count for e in ops)
    linear = sum(e.count for e in ops if e.key == "aten::linear")
    print(f"{step.__name__}: {total} ops, {linear} of them linear")
# slow_step: 3591 ops, 256 of them linear
# fast_step: 10 ops, 1 of them linear

`key_averages()` groups the records by operation name; `aten::` is the prefix of PyTorch’s built-in operations. On a GPU you add `ProfilerActivity.CUDA` and get a time for every kernel too. The usual pattern looks like this:

In [ ]:
from torch.profiler import record_function

acts = [ProfilerActivity.CPU, ProfilerActivity.CUDA]
with profile(activities=acts) as prof:
    with record_function("train_step"):   # a label in the trace
        loss = model(inputs, targets)
        loss.backward()
        optimizer.step()
print(prof.key_averages().table(sort_by="cuda_time_total",
                                row_limit=10))
prof.export_chrome_trace("trace.json")

### MFU: how much of the GPU you use

Here $N$ is the number of parameters and the peak is the total over all your chips. Work done twice to save memory, which lesson 18.4 shows, is not counted, so MFU rewards useful work only. (PaLM’s version also adds attention’s FLOPs, which matter for long inputs.) Check it against PaLM’s own numbers: 238,300 tokens a second for a 540-billion-parameter model on 6,144 TPU v4 chips, Google’s own AI chips, at 275 trillion FLOPs a second each.

In [ ]:
def mfu(tokens_per_s, n_params, peak_flops):
    return 6 * n_params * tokens_per_s / peak_flops

print(f"{mfu(238.3e3, 540e9, 6144 * 275e12):.1%}")    # 45.7%

# tokens/s a 124M model needs for 40% MFU on one H100 SXM
print(f"{0.4 * 989e12 / (6 * 124e6):,.0f}")           # 531,720

## 18.4 Memory anatomy: what fills an 80 GB GPU, and activation checkpointing

Training memory holds four things: the weights, their gradients, the optimizer’s own numbers and the saved activations. Knowing how big each is decides how to split a run, and recomputing activations instead of storing them saves the most.

[Read the lesson](https://aicourses.dev/research/pretraining-at-scale/memory-anatomy)

### Sixteen bytes per parameter

You can see the pattern in PyTorch on any computer. This small FP32 model takes one AdamW step:

In [ ]:
import torch

torch.manual_seed(0)
model = torch.nn.Sequential(
    torch.nn.Linear(1024, 4096), torch.nn.GELU(),
    torch.nn.Linear(4096, 1024))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

x = torch.randn(64, 1024)
model(x).pow(2).mean().backward()
opt.step()

def mb(tensors):
    return sum(t.numel() * t.element_size() for t in tensors) / 1e6

params = list(model.parameters())
state = [s for p in params for k, s in opt.state[p].items()
         if k != "step"]
print(f"weights:   {mb(params):.1f} MB")           # 33.6 MB
print(f"gradients: {mb(p.grad for p in params):.1f} MB")  # 33.6 MB
print(f"AdamW:     {mb(state):.1f} MB")            # 67.1 MB
print(sorted(opt.state[params[0]].keys()))
# ['exp_avg', 'exp_avg_sq', 'step']

### Activation checkpointing: recompute instead of store

PyTorch does it with `torch.utils.checkpoint.checkpoint`. To see the saving, this counts the bytes autograd keeps for the backward pass, leaving out the weights, for eight MLP blocks:

In [ ]:
from torch.utils.checkpoint import checkpoint

blocks = [torch.nn.Sequential(
    torch.nn.Linear(1024, 4096), torch.nn.GELU(),
    torch.nn.Linear(4096, 1024)) for _ in range(8)]
weights = {p.data_ptr() for b in blocks for p in b.parameters()}
x = torch.randn(256, 1024, requires_grad=True)

def activation_mb(use_checkpoint):
    saved = []
    def pack(t):                 # called for every tensor kept
        if t.data_ptr() not in weights:
            saved.append(t.numel() * t.element_size())
        return t
    with torch.autograd.graph.saved_tensors_hooks(pack, lambda t: t):
        h = x
        for block in blocks:
            h = (checkpoint(block, h, use_reentrant=False)
                 if use_checkpoint else block(h))
    return sum(saved) / 1e6

print(f"store everything: {activation_mb(False):.1f} MB")  # 75.5 MB
print(f"checkpointing:    {activation_mb(True):.1f} MB")   # 8.4 MB

## 18.5 Data parallelism, collectives, ZeRO and FSDP

The simplest way to use many GPUs is to give each a copy of the model and a different slice of the data, then average their gradients. When the copies no longer fit, FSDP splits the model’s numbers across the GPUs instead.

[Read the lesson](https://aicourses.dev/research/pretraining-at-scale/data-parallel-and-fsdp)

### Copies of the model, slices of the data

The averaging is exact. The loss of a batch is the mean over its examples, so its gradient is the mean of the examples’ gradients, and the mean of four equal-sized slices’ means is the mean of the whole. Check it on one computer:

In [ ]:
import torch

torch.manual_seed(0)
model = torch.nn.Linear(4, 1)
x, y = torch.randn(8, 4), torch.randn(8, 1)

def grad(xs, ys):
    model.zero_grad()
    torch.nn.functional.mse_loss(model(xs), ys).backward()
    return model.weight.grad.clone()

full = grad(x, y)                     # one GPU, all 8 examples
parts = [grad(x[i:i + 2], y[i:i + 2]) for i in range(0, 8, 2)]
average = sum(parts) / 4              # four GPUs, 2 examples each
print(full)     # tensor([[ 1.2357,  0.6630, -2.0608, -0.4541]])
print(average)  # tensor([[ 1.2357,  0.6630, -2.0608, -0.4541]])
print(torch.allclose(full, average))  # True

### Collectives: how GPUs share numbers

NVIDIA’s library **NCCL** (said “nickel”) runs these collectives on GPUs, rings among them. On a CPU, PyTorch uses a library called Gloo instead, so you can try a real all-reduce with four processes, separate running copies of one script, on Colab’s free CPU runtime:

In [ ]:
%%writefile allreduce.py
import torch
import torch.distributed as dist

dist.init_process_group("gloo")       # "nccl" on NVIDIA GPUs
rank, world = dist.get_rank(), dist.get_world_size()

grad = torch.tensor([float(rank + 1)] * 3)   # each worker's gradient
dist.all_reduce(grad, op=dist.ReduceOp.SUM)
grad /= world                                 # sum, then average
print(f"worker {rank}: {grad.tolist()}")
dist.destroy_process_group()

`torch.distributed` is PyTorch’s toolkit for many workers. `init_process_group` connects them; each has a **rank**, its number from 0, and `get_world_size` says how many there are. `all_reduce` replaces every worker’s tensor with the sum over all of them. The first line, as in lesson 12.7, saves the cell as `allreduce.py` instead of running it. In the next cell, start four copies with PyTorch’s launcher, **torchrun**, giving the runtime’s own address, 127.0.0.1, and any free port, a numbered door the workers meet at:

In [ ]:
!torchrun --nproc-per-node=4 --master-addr=127.0.0.1 \
          --master-port=29533 allreduce.py

### FSDP in PyTorch

**FSDP**, fully sharded data parallel, is PyTorch’s implementation of the ZeRO idea. Each **shard**, one GPU’s share of a tensor, lives on its GPU, like each chef keeping one quarter of a recipe book. You apply `fully_shard` to each layer, then to the whole model, and build the optimizer afterwards, so that it works on the sharded weights. This runs on four CPU workers too, and its first line saves it as `fsdp.py`:

In [ ]:
%%writefile fsdp.py
import torch
import torch.distributed as dist
from torch.distributed.device_mesh import init_device_mesh
from torch.distributed.fsdp import fully_shard

dist.init_process_group("gloo")       # "nccl" on NVIDIA GPUs
rank = dist.get_rank()
mesh = init_device_mesh("cpu", (dist.get_world_size(),))
torch.manual_seed(0)                  # same starting model everywhere
model = torch.nn.Sequential(
    torch.nn.Linear(64, 256), torch.nn.GELU(),
    torch.nn.Linear(256, 64))
for layer in (model[0], model[2]):
    fully_shard(layer, mesh=mesh)     # shard each layer...
fully_shard(model, mesh=mesh)         # ...then the whole model
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

w = model[0].weight
print(f"worker {rank}: full {tuple(w.shape)}, "
      f"stored here {tuple(w.to_local().shape)}")
torch.manual_seed(rank)               # different data per worker
model(torch.randn(32, 64)).pow(2).mean().backward()
opt.step()
dist.destroy_process_group()

## 18.6 Splitting one model across GPUs: tensor and pipeline parallelism

When one layer is too big for a GPU, tensor parallelism splits each matrix across several. When the model is too deep, pipeline parallelism puts different layers on different GPUs, like an assembly line.

[Read the lesson](https://aicourses.dev/research/pretraining-at-scale/model-parallelism)

### Tensor parallelism: one matrix, many GPUs

The trick, from NVIDIA’s Megatron-LM paper of 2019, is to use both: columns for $A$, rows for $B$. Then the GPUs work alone through the whole MLP and talk once, at the end. Attention splits the same way, a few heads per GPU, with one all-reduce after its output matrix. The paper counts two all-reduces per layer in the forward pass and two in the backward pass. You can check the maths with two pretend GPUs:

In [ ]:
import torch

torch.manual_seed(0)
x = torch.randn(4, 8)            # 4 tokens, hidden size 8
A = torch.randn(8, 32)           # first MLP matrix
B = torch.randn(32, 8)           # second MLP matrix
gelu = torch.nn.functional.gelu

full = gelu(x @ A) @ B           # what one GPU would compute

# two "GPUs": A split by columns, B split by rows
A0, A1 = A[:, :16], A[:, 16:]
B0, B1 = B[:16], B[16:]
part0 = gelu(x @ A0) @ B0        # GPU 0, no talking needed
part1 = gelu(x @ A1) @ B1        # GPU 1, no talking needed
print(torch.allclose(full, part0 + part1, atol=1e-5))  # True

# splitting A by rows instead breaks: GELU of a sum is not
# the sum of the GELUs
wrong = gelu(x[:, :4] @ A[:4]) + gelu(x[:, 4:] @ A[4:])
print(torch.allclose(gelu(x @ A), wrong, atol=1e-5))   # False

## 18.7 Long inputs and many experts, and choosing a layout

Context parallelism splits very long inputs across GPUs, and expert parallelism spreads a mixture of experts. Real runs combine several kinds, and the best mix depends on the hardware: Llama 3 405B used four, DeepSeek-V3 left one out.

[Read the lesson](https://aicourses.dev/research/pretraining-at-scale/long-inputs-and-experts)

### Context parallelism: one long input, many GPUs

In [ ]:
# causal attention: chunk c of the sequence attends to chunks 0..c
cp = 4
chunks = 2 * cp
work = [c + 1 for c in range(chunks)]   # key chunks each one reads

naive = [work[2 * r] + work[2 * r + 1] for r in range(cp)]
paired = [work[r] + work[chunks - 1 - r] for r in range(cp)]
print("two neighbouring chunks:", naive)    # [3, 7, 11, 15]
print("chunk r and its mirror: ", paired)   # [9, 9, 9, 9]

### Choosing a layout

Why not plain FSDP over all the GPUs? Work it out:

In [ ]:
def states_gb(n_params, tp=1, pp=1, dp=1, shard_weights=True):
    local = n_params / (tp * pp)   # this GPU's slice of the model
    weights = 2 * local / (dp if shard_weights else 1)
    rest = 14 * local / dp         # gradients + AdamW, sharded
    return (weights + rest) / 1e9

n = 405e9
print(f"{states_gb(n):.0f} GB")                    # 6480 GB, one GPU
print(f"{states_gb(n, dp=8192):.2f} GB")           # 0.79 GB, FSDP only
print(f"{states_gb(n, 8, 16, 64, False):.1f} GB")  # 7.0 GB, Llama 3

seqs = 64 * 32      # Table 4: 64 groups, 32 sequences each
print(seqs, seqs * 8192)    # 2048 16777216

## 18.8 Beyond AdamW: Muon, and keeping big runs stable

At very large scale, AdamW is no longer the only choice: Muon, a newer optimizer, has trained some of the largest open models, such as Kimi K2. Big runs also suffer sudden loss spikes, and two small changes, QK-norm and z-loss, help keep them stable.

[Read the lesson](https://aicourses.dev/research/pretraining-at-scale/optimizers-at-scale)

### Muon: an optimizer for whole matrices

where $X^\top$ is $X$ turned on its side and $a = 3.4445$, $b = -4.7750$, $c = 2.0315$ are the numbers from Jordan’s post. Every term keeps the matrix’s pieces and only changes their strengths: $(XX^\top)X$ cubes each strength and $(XX^\top)^2 X$ raises it to the fifth power, so a piece of strength $\sigma$ becomes $a\sigma + b\sigma^3 + c\sigma^5$. Worked example: $\sigma = 0.1$ becomes $0.34445 - 0.00478 + 0.00002 \approx 0.34$, more than three times stronger after one step, while strengths near 1 stay near 1.

In [ ]:
import torch

def orthogonalize(G, steps=5):
    # Newton-Schulz with the coefficients from Keller Jordan's post
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G / (G.norm() + 1e-7)        # scale so strengths are <= 1
    tall = G.shape[0] > G.shape[1]
    if tall:
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        X = a * X + (b * A + c * A @ A) @ X
    return X.T if tall else X

torch.manual_seed(0)
U, _ = torch.linalg.qr(torch.randn(6, 6))
V, _ = torch.linalg.qr(torch.randn(4, 4))
S = torch.diag(torch.tensor([10.0, 3.0, 1.0, 0.1]))
G = U[:, :4] @ S @ V.T        # a 6x4 matrix with known strengths

print(torch.linalg.svdvals(G))
# tensor([10.0000,  3.0000,  1.0000,  0.1000])
print(torch.linalg.svdvals(orthogonalize(G)))
# tensor([0.9263, 0.7525, 0.6849, 0.6842])

### Why big runs spike

**QK-norm** normalizes each query and each key, for example with **RMSNorm** (dividing a vector by the square root of its average squared entry), just before the dot product. A normalized vector cannot grow, so neither can the score, however big the weights get:

In [ ]:
torch.manual_seed(0)
d = 64
q, k = torch.randn(d), torch.randn(8, d)     # one query, 8 keys

def rms_norm(x):
    return x / x.pow(2).mean(-1, keepdim=True).sqrt()

for scale in (1, 10, 100):       # weights growing during training
    raw = (scale * q) @ (scale * k).T / d**0.5
    normed = rms_norm(scale * q) @ rms_norm(scale * k).T / d**0.5
    print(scale, f"{raw.softmax(-1).max():.3f}",
          f"{normed.softmax(-1).max():.3f}")
# 1 0.273 0.246
# 10 1.000 0.246
# 100 1.000 0.246

Here $z_i$ are the logits for one position, $Z$ is softmax’s normalizer from Module 6, and $10^{-4}$ is PaLM’s weight. It is zero only when $\log Z = 0$, so it gently pulls the logits back. In the code, `torch.logsumexp` computes $\log Z$ in one stable step:

In [ ]:
import torch.nn.functional as F

target = torch.tensor([0])
for logits in ([2.0, 1.0, 0.0], [102.0, 101.0, 100.0]):
    z = torch.tensor([logits])
    ce = F.cross_entropy(z, target)
    log_Z = torch.logsumexp(z, dim=-1)
    z_loss = 1e-4 * log_Z**2
    print(f"loss {ce.item():.4f}  log Z {log_Z.item():.4f}  "
          f"z-loss {z_loss.item():.6f}")
# loss 0.4076  log Z 2.4076  z-loss 0.000580
# loss 0.4076  log Z 102.4076  z-loss 1.048732

## 18.9 Fitting scaling laws across model sizes

You train several small models at each fixed compute budget and fit the curve properly, with error bars. It is the experiment labs run before betting on a big model, and the centre of Stanford’s CS336 scaling assignment.

[Read the lesson](https://aicourses.dev/research/pretraining-at-scale/scaling-law-fits)

### Fitting it yourself

You will fit real runs in this module’s project. To learn the method, start from simulated ones: losses computed from a published scaling law, Epoch AI’s 2024 refit of Chinchilla’s formula, plus a little noise. Because you know the law, you can check that your fit finds it: minimizing this formula at fixed $C$ gives an exponent of $\beta / (\alpha + \beta) = 0.3658 / 0.7136 \approx 0.513$, where $\alpha$ and $\beta$ are the powers on $N$ and $D$ in the code below.

In [ ]:
import numpy as np

def true_loss(N, D):      # Epoch AI's refit of Chinchilla's law
    return 1.8172 + 482.01 / N**0.3478 + 2085.43 / D**0.3658

rng = np.random.default_rng(0)
budgets = [1e18, 3e18, 1e19, 3e19]          # FLOPs per run
runs = []                                   # (C, N, loss)
for C in budgets:
    for N in np.geomspace(2e7, 2e9, 9):     # 20M to 2B params
        D = C / (6 * N)                     # tokens the budget buys
        loss = true_loss(N, D) + rng.normal(0, 0.01)
        runs.append((C, N, loss))
print(len(runs), "runs")                    # 36 runs

`np.geomspace` spaces the nine sizes evenly on a log scale. Four budgets, nine sizes each: 36 runs, each with random noise of about 0.01 added to its loss. Now find each valley and the line through them:

In [ ]:
def best_size(runs, C):
    pts = [(np.log10(N), L) for c, N, L in runs if c == C]
    x, y = np.array(pts).T
    p, q, _ = np.polyfit(x, y, 2)          # a parabola per budget
    return 10 ** (-q / (2 * p))            # its lowest point

def fit(runs):
    best = [best_size(runs, C) for C in budgets]
    slope, icpt = np.polyfit(np.log10(budgets), np.log10(best), 1)
    return slope, icpt

slope, icpt = fit(runs)
for C in budgets:
    print(f"{C:.0e} FLOPs: best size {best_size(runs, C):.3g}")
print(f"N_opt grows as C^{slope:.3f}")
print(f"prediction at 1e21: {10 ** (icpt + 21 * slope):.3g}")
# 1e+18 FLOPs: best size 8.03e+07
# 3e+18 FLOPs: best size 1.39e+08
# 1e+19 FLOPs: best size 2.65e+08
# 3e+19 FLOPs: best size 4.47e+08
# N_opt grows as C^0.509
# prediction at 1e21: 2.69e+09

### Error bars: how sure is the line?

One fit gives one number. Rerun the experiment with different noise and you would get a slightly different one. The **bootstrap** from Module 12 estimates how much: draw a new set of runs from your own runs at random, with repeats allowed, refit, and do it a thousand times. The spread of the answers is your uncertainty.

In [ ]:
slopes, preds = [], []
for _ in range(1000):
    sample = []
    for C in budgets:                  # resample within each budget
        mine = [r for r in runs if r[0] == C]
        pick = rng.integers(0, len(mine), len(mine))
        sample += [mine[i] for i in pick]
    s, i = fit(sample)
    slopes.append(s)
    preds.append(10 ** (i + 21 * s))
lo, hi = np.percentile(slopes, [5, 95])
print(f"exponent {slope:.3f}, 90% interval {lo:.3f} to {hi:.3f}")
lo, hi = np.percentile(preds, [5, 95])
print(f"size at 1e21: {lo:.3g} to {hi:.3g}")
# exponent 0.509, 90% interval 0.475 to 0.517
# size at 1e21: 2.27e+09 to 2.81e+09

## 18.10 Project: from one GPU to eight, with a scaling curve

Shows you can check parallel training code, measure a GPU and predict a run beyond one GPU from first principles, the core of pretraining work.

[Read the lesson](https://aicourses.dev/research/pretraining-at-scale/project-multi-gpu-run)

**This is the project.** These cells are starting points from the lesson; the steps between them are yours to write. A cell that uses a name from one of your steps stops with a NameError until you have written that step.

### Steps

1. **Check the parallel code on CPU workers.** In Colab’s free CPU runtime, shard your GPT as in lesson 18.5, give each worker its own slice of every batch, and switch dropout off, so the only difference between runs is the split. Start from this tiny version:

In [ ]:
%%writefile same_result.py
import torch, torch.distributed as dist
from torch.distributed.device_mesh import init_device_mesh
from torch.distributed.fsdp import fully_shard
dist.init_process_group("gloo")
rank, world = dist.get_rank(), dist.get_world_size()
torch.manual_seed(0)                      # same model and data
model = torch.nn.Sequential(torch.nn.Linear(16, 64),
                            torch.nn.GELU(), torch.nn.Linear(64, 1))
x, y = torch.randn(64, 16), torch.randn(64, 1)   # one batch of 64
mesh = init_device_mesh("cpu", (world,))
for part in (model[0], model[2], model):
    fully_shard(part, mesh=mesh)          # layers, then the model
opt = torch.optim.AdamW(model.parameters(), lr=1e-2)
mine = slice(rank * 64 // world, (rank + 1) * 64 // world)
for step in range(20):                    # each trains on its slice
    loss = (model(x[mine]) - y[mine]).pow(2).mean()
    loss.backward()
    opt.step()
    opt.zero_grad()
with torch.no_grad():                     # score the whole batch
    full = (model(x) - y).pow(2).mean()
if rank == 0:
    print(f"{world} worker(s): loss {full.item():.6f}")
dist.destroy_process_group()